##### Library imports

In [ ]:
import re, os, sys
import pandas as pd
import cv2 as cv
import SimpleITK as sitk
import numpy as np 
from skimage import measure, filters
import json
import matplotlib.pyplot as plt

##### Function definitions

In [ ]:
def window_stack_sitk(input_im, window_center=40, window_width=80):
    """
    Clamps values outside [min, max] to the edge values.
    
    Parameters
    ----------
    input_im : sitk.Image
        Input, unwindowed, brain image
    window_center : int, optional
        The center of the Hounsfield Unit (HU) scale in the windowed image. Default is 40 HU (brain).
    window_width : int, optional
        The total HU window width around the window_center, in the windowed image. Default is 80 HU (brain).

    Returns
    -------
    windowed_im : sitk.Image
        Windowed CT image with the desired tissue's visualization optimized (e.g., 0 - 80 HU for brain tissue)
    
    """
    img_min = window_center - (window_width // 2)
    img_max = window_center + (window_width // 2)


    windowed_im = sitk.IntensityWindowing(
        input_im, 
        windowMinimum=float(img_min), 
        windowMaximum=float(img_max), 
        outputMinimum=float(img_min), 
        outputMaximum=float(img_max)
    )
    
    return windowed_im

In [ ]:
def getLargestCC(blobs_labels):
    """Returns the largest connected component from an image containing blobs of discrete labels
    
    Parameters
    ----------
    blobs_labels : numpy.ndarray 
        A collection of discrete blobs (e.g., cerebrospinal fluid [CSF] spaces including the lateral ventricles, longitudinal fissure, sulci etc.)

    Returns
    -------
    largestCC: numpy.ndarray
        The largest connected component from a collection of discrete blobs (e.g. lateral ventricles from a collection of CSF spaces)  
    
    """
    # assume at least 1 CC apart from background
    if blobs_labels.max() == 0:
        raise ValueError('Blank segmentation, inspect processing up to here.')
    #this assumes that the background is the largest CC (label 0)
    largestCC = blobs_labels == np.argmax(np.bincount(blobs_labels.flat)[1:])+1 
    return largestCC

In [ ]:
def rotate_image(image, dimension = 3):
    """Rotates NIfTI axial images so they are upright for visualization. Simple insights tool kit (SITK) utilizes the LPS coordinate system, 
       meaning image voxel coordinates are assumed to increase in the Right --> Left, Anterior --> Posterior, and Inferior --> Superior directions. 
       However, NIfTI images use the RAS coordinate system, which makes axial slices (lateral cross-sections) appear inverted when read with SITK. 
       This function resamples the image such that it is visualized in upright direction, enabling interpretable visualization and processing. 
    
    Parameters
    ----------
    image : sitk.Image
        The original NIfTI image (Note that the direction cosine should be [-1,0,0,0,-1,0,0,0,1])
    dimension : int, optional
        Dimensionality of the original and rotation-corrected image. Default is 3.
    
    Returns
    -------
    rotated_image : sitk.Image
        Rotation-corrected image, with a direction cosine of [1,0,0,0,1,0,0,0,1]
    """

    transform = sitk.AffineTransform(dimension)
    transform.SetCenter(image.TransformContinuousIndexToPhysicalPoint(np.array(image.GetSize())//2.0))
    s = image.GetSpacing()[2]

    matrix = np.array([[1.0,0.0,0.0],[0.0,1.0,0.0],[0.0,0.0,1.0]])

   
    transform.SetMatrix(matrix.ravel())

    extreme_points = [image.TransformIndexToPhysicalPoint((0,0,0)), 
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),0,0)),
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),image.GetHeight(),0)),
                      image.TransformIndexToPhysicalPoint((0,image.GetHeight(),0)),
                      image.TransformIndexToPhysicalPoint((0,0,image.GetDepth())), 
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),0,image.GetDepth())),
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),image.GetHeight(),image.GetDepth())),
                      image.TransformIndexToPhysicalPoint((0,image.GetHeight(),image.GetDepth()))]

    inv_transform = transform.GetInverse()

    extreme_points_transformed = [inv_transform.TransformPoint(pnt) for pnt in extreme_points]
    min_x = min(extreme_points_transformed)[0]
    min_y = min(extreme_points_transformed, key=lambda p: p[1])[1]
    min_z = min(extreme_points_transformed, key=lambda p: p[2])[2]
    max_x = max(extreme_points_transformed)[0]
    max_y = max(extreme_points_transformed, key=lambda p: p[1])[1]
    max_z = max(extreme_points_transformed, key=lambda p: p[2])[2]

    # Use the original spacing (arbitrary decision).
    output_spacing = image.GetSpacing()
    # Identity cosine matrix.   
    output_direction = [1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0]
    # Minimal x,y coordinates are the new origin.
    output_origin = [min_x, min_y, min_z]
    
    # Compute grid size based on the physical size and spacing.
    output_size = image.GetSize()
    
    rotated_image = sitk.Resample(image, output_size, transform, sitk.sitkLinear, output_origin, output_spacing,
                                  output_direction)
    return rotated_image

In [ ]:
def moving_average(x, w):
    """
    Computes the 1D moving average of an array using a specified window size, 
    preserving the original array's length through zero-padding.
    
    Parameters
    ----------
    x : array_like
        The input 1D array to be smoothed. 
        Note: The array is cast to `uint8` prior to calculation.
    w : int
        The size of the moving average window.

    Returns
    -------
    numpy.ndarray
        A 1D array of floats representing the smoothed moving average of the input.
    """
    
    # Cast input to 8-bit unsigned integer
    x = np.uint8(x)
    
    # Calculate padding to ensure the output length matches the input length.
    # This automatically handles both even and odd window sizes (w).
    pad_left = w // 2
    pad_right = (w - 1) // 2
    
    # Apply zero-padding
    zero_pad_x = np.pad(x, (pad_left, pad_right), mode='constant', constant_values=(0, 0))
    
    # Compute the moving average using a valid linear convolution
    return np.convolve(zero_pad_x, np.ones(w), 'valid') / w

In [ ]:
def get_fm_row(ventricles, aj):
    """
    Determines a specific anatomical row marker (e.g., Foramen of Monro) by 
    analyzing the smoothed vertical profile of the largest connected ventricle.
    
    Parameters
    ----------
    ventricles : numpy.ndarray
        A 2D binary array representing the segmented ventricles.
    aj : float or int
        The starting row index (e.g., Anterior Commissure Y-coordinate) used 
        as a baseline to begin the search.

    Returns
    -------
    fm_row : int
        The row index where the ventricle profile stops decreasing (local minimum), 
        searched from an offset of `aj + 5`.
    col_profile : numpy.ndarray
        A 1D array representing the moving average of the horizontal pixel 
        counts for each row.
    """
    
    # Isolate the largest connected component
    blobs_labels = measure.label(ventricles, background=0)
    largest_cc = getLargestCC(blobs_labels)

    # Calculate a smoothed vertical profile (sum of foreground pixels per row)
    # using a window size of 3
    col_profile = moving_average(np.sum(largest_cc, axis=1), 3)
    
    # Define the starting index for our search (offset by 5 rows from aj)
    start_idx = int(np.round(aj + 5))
    
    try:
        # np.diff >= 0 finds where the profile stops decreasing (starts increasing or goes flat)
        # We search only from start_idx downwards
        local_min_idx = np.where(np.diff(col_profile[start_idx:]) >= 0)[0][0]
        fm_row = local_min_idx + start_idx
        
    except IndexError:
        # Fallback in case the ventricle profile tapers off without ever increasing again
        print(f"[Warning] No local minimum found in the profile after row {start_idx}. Defaulting to start index.")
        fm_row = start_idx

    return fm_row, col_profile

In [ ]:
def calc_nmax_3vw(axial_data, ak, aj, pj, nmax_3vw_processing_params):
    """
    Calculates the normalized maximum width of the third ventricle (3VW).

    This function scans through 10mm of axial slices above the AC-PC line. It isolates 
    the ventricles using a combination of Gaussian blurring, Otsu's method, and adaptive 
    mean thresholding. It creates specific Region of Interest (ROI) masks to isolate 
    the third ventricle and calculates its maximum width. The final width is normalized 
    against the total width of the brain at that specific slice. If the algorithm fails 
    to isolate the 3V, it relaxes the anterior-posterior boundaries and retries.

    Parameters
    ----------
    axial_data : numpy.ndarray
        3D array representing the axial CT volume data.
    ak : float
        The z-axis starting coordinate (slice index) representing the AC-PC line.
    aj : float
        The y-axis (row) coordinate representing the anterior commissure (AC).
    pj : float
        The y-axis (row) coordinate representing the posterior commissure (PC).
    nmax_3vw_processing_params : dict
        Dictionary containing processing thresholds and kernel sizes.

    Returns
    -------
    n_3w_width : float or None
        The normalized maximum width of the third ventricle. Returns None if 
        the region cannot be found within the maximum threshold iterations.
    """

    cur_thresh = nmax_3vw_processing_params["cur_thresh"]
    dec_thresh = nmax_3vw_processing_params["dec_thresh"]
    ax_gauss_blur_size = nmax_3vw_processing_params["ax_gauss_blur_size"]
    ax_adaptive_thresh_size = nmax_3vw_processing_params["ax_adaptive_thresh_size"]
    ax_adaptive_thresh_mean_thresh = nmax_3vw_processing_params["ax_adaptive_thresh_mean_thresh"]
    v3_roi_col_left_lim_dec = nmax_3vw_processing_params["v3_roi_col_left_lim_dec"]
    v3_roi_col_right_lim_inc = nmax_3vw_processing_params["v3_roi_col_right_lim_inc"]
    
    third_v_width = []
    max_3vw = 0 
    
    #Scan slices 10 mm above AC-PC line
    while True:        
        for plane in range(int(round(ak)), int(round(ak))+10):
            axial_brain_im = axial_data[plane,:,:]

            #initial gauss blur
            axial_gauss_blurred = cv.GaussianBlur(np.uint8(axial_brain_im),(ax_gauss_blur_size,ax_gauss_blur_size), cv.BORDER_DEFAULT)

            thresh = filters.threshold_otsu(axial_gauss_blurred)

            # axial_brain_binary = axial_gauss_blurred > thresh   

            #Adaptive threshold
            axial_ad_th = cv.adaptiveThreshold(axial_gauss_blurred, 1, cv.ADAPTIVE_THRESH_MEAN_C,
                                            cv.THRESH_BINARY, ax_adaptive_thresh_size, ax_adaptive_thresh_mean_thresh)

            # The background needs to be subtracted from the adaptive threshold because background pixels are classified as foreground 
            # because of local thresholding
            blobs_labels = measure.label(axial_brain_im, background = 0)
            background_pixels = blobs_labels == np.argmax(np.bincount(blobs_labels.flatten()))
            
            axial_brain_mask = np.where(background_pixels>0,0, axial_ad_th)

            blobs_labels = measure.label(axial_brain_mask, background=0)
            ventricle_mask = getLargestCC(blobs_labels)

            im_th = ventricle_mask.copy()
            h, w = im_th.shape[:2]
            mask = np.zeros((h+2, w+2), np.uint8)
            im_floodfill = im_th.copy()
            # Floodfill from point (0, 0)
            res = cv.floodFill(np.uint8(im_floodfill), mask, (0,0), 255)
            full_brain_mask = (1-res[2]).copy()
            full_brain_mask = full_brain_mask[1:h+1,1:w+1]
            

            axial_brain_im = np.where(full_brain_mask > 0, axial_brain_im, 0)
            ventricles = full_brain_mask - ventricle_mask

            v3_roi = np.zeros(ventricles.shape)
            [r_v, c_v] = ventricles.nonzero()

            if len(r_v) == 0:
                continue

            mr_v, mc_v = (max(r_v) + min(r_v))//2, (max(c_v) + min(c_v))//2

            left_col = max(0, mc_v - v3_roi_col_left_lim_dec)
            right_col = min(v3_roi.shape[1], mc_v + v3_roi_col_right_lim_inc)
            v3_roi[int(round(aj)):int(round(pj)),left_col:right_col] = 1
            ventricles_connected = np.logical_or(ventricles,v3_roi)

            #Here, we obtain a better starting point to the 3V ROI as opposed to the AC because, lower parts of the 
            #frontal horns might be captured with an ROI starting at the AC. The logical OR with the ROI and the ventricles
            #makes the column profile more uniform in the region we need. The default offset is 5 mm behind the AC. 
            #But if the column profile does not start to decrease at that point, we find the first point where it happens
            
            #In case the ROI we define is smaller than the 3V in width, and the AC lies slightly ahead of the foramen
            #(where the frontal horns end), the first point where this signal decreases might be more anterior than where
            #we need it to be. That is why we have a 5 mm offset to help it
            
            try:
                fm_row, col_profile = get_fm_row(ventricles_connected, aj)
            except:
                continue

            #Again, we have a default offset of 3 mm to cap the 3v vertically, but we decrease it if needed. This gives us starting and ending
            #points that are slightly posterior to the AC and slightly anterior to the PC. 
            fh_roi = np.zeros(ventricles_connected.shape)
            fh_start = max(0, fm_row + (cur_thresh - dec_thresh))
            fh_end = max(0, int(round(pj)) - (cur_thresh - dec_thresh))
            
            fh_roi[fh_start:fh_end,:] = 1 

            ventricles_connected_2 = fh_roi*ventricles_connected

            try:
                [r,c] = (1-getLargestCC(measure.label(ventricles_connected_2, background = 0))).nonzero()
            except:
                continue

            ventricles[r,c] = 0

            [tv_r, tv_c] = ventricles.nonzero()

            if len(tv_r) == 0:
                continue
                
            temp_3v = []
            for r in sorted(np.unique(tv_r)):
                temp_3v = temp_3v + [max(tv_c[tv_r == r]) - min(tv_c[tv_r == r])]
                
            third_v_width = np.median(np.array(temp_3v))
            if third_v_width > max_3vw:
                max_3vw = third_v_width
                opt_pl = plane
                opt_ven = ventricles
                full_brain_mask_opt = full_brain_mask
                axial_brain_im_opt = axial_brain_im   
        if max_3vw > 0:               

            [row, col] = axial_brain_im_opt[axial_brain_im_opt.shape[0]//2:,:].nonzero()
            sorted_unique_rows = sorted(np.unique(row))
            opt_ri = sorted_unique_rows[np.argmax(np.array([max(col[row==r]) - min(col[row==r])
                                                            for r in sorted_unique_rows]))]
            brain_l_x, brain_r_x = min(col[row==opt_ri]),max(col[row==opt_ri])
            brain_l_y, brain_r_y = opt_ri + axial_brain_im_opt.shape[0]//2, opt_ri + axial_brain_im_opt.shape[0]//2

            [tv_r, tv_c] = opt_ven.nonzero()
            n_3w_width = max_3vw/(brain_r_x - brain_l_x)  
            
            fig, ax = plt.subplots(figsize = (10,10))
            ax.imshow(axial_brain_im_opt,cmap = 'gray')
            [ax.scatter(x,y,marker ='x',color = 'w', s = 2) 
             for x,y in zip(range(min(tv_c),max(tv_c)+1), [(min(tv_r)+max(tv_r))//2]*(max(tv_c)-min(tv_c)+1))]
            #offset y coordinate so it does not overlap with the 3vw
            [ax.scatter(x,y + 5,marker ='x',color = 'w', s= 2) 
             for x,y in zip(range(brain_l_x,brain_r_x+1), [brain_l_y]*(brain_r_x-brain_l_x+1))]                          
            plt.show()
            
            return n_3w_width
        else:
            dec_thresh = dec_thresh + 1
            if dec_thresh > 2:
                print('Unable to calculate NMax-3VW as 3V ROI is going beyond the Foramen of Monro (AC) row or PC, returning None')
                return None

##### Data setup

###### Before running the pipeline, make a copy of config.template.json, rename it to config.json, and update the paths to point to your local dataset.

In [ ]:
# Load the configuration file
with open('config.json', 'r') as file:
    config = json.load(file)

# Assign variables dynamically
data_path = config['data_path']
axial_acpc_vol_aligned_name = config['axial_acpc_vol_aligned_name']
info_csv_path = config['info_csv_path_w_name']
scan_id_col_name = config['scan_id_col_name']
pat_id_col_name = config['pat_id_col_name']
ac_coordinates_col_name = config['ac_coordinates_col_name']
pc_coordinates_col_name = config['pc_coordinates_col_name']

output_folder = config['output_csv_write_folder']

print(f"Loading data from: {data_path}")

In [ ]:
info_df = pd.read_csv(info_csv_path)
info_df[scan_id_col_name] = info_df[scan_id_col_name].str.strip("'")

In [ ]:
accnum_list = info_df[scan_id_col_name].values

In [ ]:
len(accnum_list), len(info_df)

In [ ]:
#dataframe which will contain track of the computed NMax-3VW values for each scan
third_ven_max_width_df = pd.DataFrame()

###### Predefined, emperically determined processing parameters corresponding to ROI sizes, positions, and global/local thresholding parameters. 
The following parameters (ei_calc_processing_params) work for a diverse cohort of chronic neurodegenerative conditions spanning Normal Pressure Hydrocephalus, Alzheimer's disease, post-traumatic encephalomalacia, and headache. So they are recommended to be used directly for patient scans without acute pathology like significant midline shifts, bleeds, or mass effects. 

In [ ]:
nmax_3vw_processing_params = dict()
nmax_3vw_processing_params["cur_thresh"] = 3
nmax_3vw_processing_params["dec_thresh"] = 0
nmax_3vw_processing_params["ax_gauss_blur_size"] = 3
nmax_3vw_processing_params["ax_adaptive_thresh_size"] = 85
nmax_3vw_processing_params["ax_adaptive_thresh_mean_thresh"] = 7.5
nmax_3vw_processing_params["v3_roi_col_left_lim_dec"] = 7
nmax_3vw_processing_params["v3_roi_col_right_lim_inc"] = 8

##### NMax-3VW pipeline

In [ ]:
for acc_num in accnum_list:
   
    pat = info_df[pat_id_col_name][info_df[scan_id_col_name]==acc_num].values[0]
        
    pca_str = info_df[pc_coordinates_col_name][(info_df[scan_id_col_name]==acc_num)].values[0]
    [x,y,z] = pca_str.split(",")
    pca_x,pca_y,pca_z = [float(x.strip("[")),float(y),float(z.strip("]"))]      
    aca_str = info_df[ac_coordinates_col_name][(info_df[scan_id_col_name]==acc_num)].values[0]
    [x,y,z] = aca_str.split(",")
    aca_x,aca_y,aca_z = [float(x.strip("[")),float(y),float(z.strip("]"))]   
            

    study_path = os.path.join(data_path,
                              acc_num, axial_acpc_vol_aligned_name)
  
    ax_img = sitk.ReadImage(study_path)
    ax_img_brain = window_stack_sitk(rotate_image(ax_img,3))
    
    ac = [aca_x,aca_y,aca_z]
    pc = [pca_x,pca_y,pca_z]
    axial_data = sitk.GetArrayFromImage(ax_img_brain) 
    N = axial_data.shape        
    ai, aj, ak = ax_img_brain.TransformPhysicalPointToContinuousIndex(ac)
    pi, pj, pk = ax_img_brain.TransformPhysicalPointToContinuousIndex(pc)
    
    n_3w_width = calc_nmax_3vw(axial_data, ak, aj, pj, nmax_3vw_processing_params)
    third_ven_max_width_df = pd.concat([third_ven_max_width_df, pd.DataFrame({pat_id_col_name:[pat],
                                                                                 scan_id_col_name:[acc_num], 
                                                                                 'nmax_3vw':[n_3w_width]
                                                                                     })], 
                                 ignore_index = True)
       
        

In [ ]:
len(third_ven_max_width_df), sum(third_ven_max_width_df['nmax_3vw'].isna())

In [ ]:
#Examine histogram of computed values
plt.hist(third_ven_max_width_df['nmax_3vw'])
plt.xticks([x for x in np.arange(0, 0.14, 0.01)])
plt.show()

In [ ]:
#Check for any extreme values and inspect the corresponding outputs above for computational errors
min_3vw_lim = 0.002
max_3vw_lim = 0.2

third_ven_max_width_df[(third_ven_max_width_df['nmax_3vw'] <= min_3vw_lim) | (third_ven_max_width_df['nmax_3vw'] >= max_3vw_lim)]

In [ ]:
third_ven_max_width_df.to_csv(os.path.join(output_folder, 'nmax_3vw.csv'))